# 直接偏好优化（2）模型训练

In [1]:

import os
import pickle

from src.core import *
from src.gpt import *

In [2]:
np.random.seed(42)

In [3]:
class SFTDataset(Dataset):

    def __init__(self, filename, context_size=64, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.split = split
        super().__init__(1)

    def load(self):
        with open(self.filename, "rb") as f:
            examples = pickle.load(f)

        split = int(len(examples) * self.split)
        self.train_data = self._pack(examples[:split])
        self.test_data = self._pack(examples[split:])

    def _pack(self, examples):
        xs, ys, masks = [], [], []
        for prompt, response in examples:
            x, y, mask = self._build_example(prompt, response)
            xs.append(x)
            ys.append(y)
            masks.append(mask)
        return xs, ys, masks

    def _build_example(self, prompt, response):
        tokens = list(prompt) + list(response)
        if len(tokens) > self.context_size + 1:
            overflow = len(tokens) - (self.context_size + 1)
            prompt = prompt[overflow:] if overflow < len(prompt) else []
            tokens = list(prompt) + list(response)
            tokens = tokens[-(self.context_size + 1):]

        response_start = len(prompt)

        x = np.array(tokens[:-1], dtype=np.int64)
        y = np.array(tokens[1:], dtype=np.int64)
        mask = np.arange(len(y)) + 1 >= response_start
        return x, y, mask

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y, mask = self.data
        return Tensor(x[s]), Tensor(y[s]), Tensor(mask[s])

In [4]:
class DPODataset(SFTDataset):

    def _pack(self, examples):
        cxs, cys, cmasks, rxs, rys, rmasks = [], [], [], [], [], []
        for prompt, chosen, rejected in examples:
            x, y, m = self._build_example(prompt, chosen)
            cxs.append(x)
            cys.append(y)
            cmasks.append(m)
            x, y, m = self._build_example(prompt, rejected)
            rxs.append(x)
            rys.append(y)
            rmasks.append(m)
        return cxs, cys, cmasks, rxs, rys, rmasks

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        cx, cy, cmask, rx, ry, rmask = self.data
        return (Tensor(cx[s]), Tensor(cy[s]), Tensor(cmask[s]),
                Tensor(rx[s]), Tensor(ry[s]), Tensor(rmask[s]))

In [5]:
class DPOLoss(Loss):

    def __init__(self, beta=0.1):
        self.beta = beta

    def loss(self, pc: Tensor, pr: Tensor, rc, rr):
        p_diff = pc.data - pr.data
        r_diff = np.asarray(rc) - np.asarray(rr)
        margin = self.beta * (p_diff - r_diff)
        sig = 1.0 / (1.0 + np.exp(-margin))
        e = Tensor(-np.sum(np.log(np.clip(sig, 1e-10, 1.0))) / pc.shape[0])

        def gradient_fn():
            common = (sig - 1.0) / pc.shape[0] * self.beta * e.grad
            pc.grad += common
            pr.grad += -common

        return e.attach(gradient_fn, parents={pc, pr})

In [6]:
class DPOModel(GPTModel):

    def __init__(self, layer, loss_fn, optimizer, reference):
        super().__init__(layer, loss_fn, optimizer)
        self.reference = reference
        self.reference.eval()

    def train(self, dataset, epochs, scheduler=None, filename=None):
        self.layer.train()

        steps = 0
        for epoch in range(epochs):
            order = list(range(len(dataset)))
            np.random.shuffle(order)

            total_loss = 0.0
            for step, i in enumerate(order):
                if scheduler is not None:
                    self.optimizer.lr = scheduler.step(steps)

                cx, cy, cm, rx, ry, rm = dataset[i]
                policy_chosen = self._sequence_log_prob(self.layer(cx), cy, cm)
                policy_rejected = self._sequence_log_prob(self.layer(rx), ry, rm)
                ref_chosen = self._sequence_log_prob(self.reference(cx), cy, cm).data
                ref_rejected = self._sequence_log_prob(self.reference(rx), ry, rm).data
                loss = self.loss_fn(policy_chosen, policy_rejected, ref_chosen, ref_rejected)

                self.optimizer.zero_grad()
                loss.backward()
                total_loss += float(loss.data)
                self.optimizer.clip_grad_norm()
                self.optimizer.step()
                steps += 1

                if (step + 1) % 100 == 0:
                    lr = f" lr {self.optimizer.lr:.6f}" if scheduler is not None else ""
                    print(f"epoch {epoch + 1} step {step + 1}/{len(dataset)} loss {(total_loss / 100):.4f}{lr}")
                    total_loss = 0.0

            if filename is not None:
                self.save(filename)
                print(f"epoch {epoch + 1} saved DPO model to {filename}")

    def test(self, dataset):
        return None

    @staticmethod
    def _sequence_log_prob(x, y, mask):
        exp = np.exp(x.data - np.max(x.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)
        y = np.asarray(y.data, dtype=np.int64)
        mask = np.asarray(mask.data, dtype=softmax.dtype)
        picked = np.clip(np.take_along_axis(softmax, y[..., None], axis=-1).squeeze(-1), 1e-10, 1)
        p = Tensor(np.sum(np.log(picked) * mask, axis=-1))

        def gradient_fn():
            grad = -softmax.copy()
            target_grad = np.take_along_axis(grad, y[..., None], axis=-1) + 1.0
            np.put_along_axis(grad, y[..., None], target_grad, axis=-1)
            grad *= mask[..., None]
            x.grad += p.grad[:, None, None] * grad

        return p.attach(gradient_fn, {x})

    def load_reference(self, filename):
        if os.path.isfile(filename):
            data = np.load(filename, allow_pickle=False)
            for i, p in enumerate(self.reference.parameters):
                p.data = data[f"param_{i}"]
                p.grad = np.zeros_like(p.data)

In [7]:
DATA_FILE = "../../../tinyshakespeare.txt"
MODEL_FILE = "../../../tinyshakespeare-gpt.npz"
DPO_SAMPLES = "../../dpo-samples.pkl"
DPO_MODEL = "../../tinyshakespeare-dpo.npz"

In [8]:
LEARNING_RATE = 0.00001
BATCH_SIZE = 4
CONTEXT_SIZE = 32
EMBEDDING_SIZE = 64
HEADS = 2
BLOCKS = 2

In [9]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
layer = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
reference = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = DPOLoss()
optimizer = AdamWOptimizer(layer.parameters, lr=LEARNING_RATE)
model = DPOModel(layer, loss_fn, optimizer, reference)
model.load(MODEL_FILE)
model.load_reference(MODEL_FILE)

In [10]:
dpo_dataset = DPODataset(DPO_SAMPLES, CONTEXT_SIZE)
scheduler = WarmupCosineScheduler(LEARNING_RATE, len(dpo_dataset), 50, LEARNING_RATE / 10)
model.train(dpo_dataset, 1, scheduler, DPO_MODEL)

epoch 1 step 100/921 loss 0.6522 lr 0.000010
epoch 1 step 200/921 loss 0.6515 lr 0.000009
epoch 1 step 300/921 loss 0.6733 lr 0.000008
epoch 1 step 400/921 loss 0.6325 lr 0.000007
epoch 1 step 500/921 loss 0.6558 lr 0.000005
epoch 1 step 600/921 loss 0.6148 lr 0.000004
epoch 1 step 700/921 loss 0.6190 lr 0.000002
epoch 1 step 800/921 loss 0.6039 lr 0.000001
epoch 1 step 900/921 loss 0.6396 lr 0.000001
epoch 1 saved DPO model to ../../tinyshakespeare-dpo.npz


In [11]:
print(model.generate(dataset, prompt="ROMEO:"))

ROMEO:
What wrow would up copincio fie,
That If I read the good them when death!

FLORIZELS:
Where should, love in villace, what bay.

PARTHUS:
Is right his uny you wone thy is ranglace I is excong.

Respootoring, Go sonesstain.

COLINIUS:
Sor jusy sentain, of EShat me's presenecanny.

VOMETHUMIO:
Grack weady drive.

SOMENESTER:
My no such good blad; and thou shall he it, that 'twourn or mone,
For backs did hearter; which; good him me butherse your thee was a not.
Mard's facet to saums your introk, dingue, not is
